# Iterative-Methods Test Engine — Walkthrough

A complete, end-to-end use of the framework for someone who wants to know **how to drive
it with their own problem and their own method**. We go from the math to a finished,
persisted, plotted comparison:

1. **State the problem** — ridge regression, and its math.
2. **Define & register a problem element** — subtype `Objective`, implement the interface, register a generator.
3. **Author a custom method** — implement the `IterativeMethod` contract (Polyak's heavy-ball).
4. **Configure an experiment** — baseline vs experimental roles, a variant grid, stopping criteria.
5. **Run it** — one call.
6. **Persist & reload** — reproducibility round-trip.
7. **Analyze & plot** — `to_dataframe` → a convergence figure.

Everything here uses the engine's **public, exported API**. The engine (`src/`) ships only
abstractions and machinery; problems and methods are *content* that extend it — which is
exactly what steps 2 and 3 do, live.

## Running this notebook

This is **clone-and-run** — the first code cell activates the project beside the notebook
and instantiates its dependencies, so you don't manage environments yourself. You only
need a Julia Jupyter kernel:

- **VS Code** — open the notebook and pick a Julia kernel (the Julia extension provides one).
- **Classic Jupyter** — IJulia ships with this project. From the repository root:
  ```bash
  julia --project -e 'using Pkg; Pkg.instantiate()'        # one-time: fetch deps (incl. IJulia)
  julia --project -e 'using IJulia; notebook(dir=pwd())'   # launches Jupyter on this folder
  ```
  Then open `walkthrough.ipynb` and select the **julia-1.12** kernel.

In [ ]:
# Clone-and-run: activate the project beside the notebook and install missing deps.
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

In [ ]:
# Load the ENGINE + existing CONTENT (problems, methods, components). We do NOT include
# the experiment scripts here — we author our own problem and method below.
include("experiments/_bootstrap.jl")   # brings TestEngine into scope via `using .TestEngine`

using LinearAlgebra
using Random
using CairoMakie
CairoMakie.activate!(type = "png")     # inline static figures
println("engine + content loaded")

## 1 · The problem you want to solve

We'll use **ridge regression** (Tikhonov-regularized least squares):

$$\min_{x\in\mathbb{R}^n}\; f(x) = \tfrac{1}{2}\lVert Ax-b\rVert_2^2 \;+\; \tfrac{\mu}{2}\lVert x\rVert_2^2 .$$

It is smooth and strongly convex, so everything is closed-form:

- gradient $\;\nabla f(x) = A^\top(Ax-b) + \mu x$
- Hessian $\;\nabla^2 f(x) = A^\top A + \mu I$ (constant, symmetric positive-definite)
- unique minimizer $\;x^\star = (A^\top A + \mu I)^{-1} A^\top b$

The Hessian's eigenvalues set the smoothness $L=\lambda_{\max}+\mu$ and strong-convexity
$m=\lambda_{\min}+\mu$ constants, hence the condition number $\kappa = L/m$ — the quantity
that governs how fast first-order methods converge. We'll build $A$ with a **controlled
spectrum** so $\lambda_{\max}(A^\top A)=1$ (so $L=1+\mu$ is known up front) and $\kappa$ is a
parameter we choose.

## 2 · Define & register the problem element

A problem family is **content**: a struct that subtypes `Objective` and implements the
three functions the engine dispatches on —

- `value(f, x)` → $f(x)$
- `grad!(g, f, x)` → write $\nabla f(x)$ into `g`, in place
- `hessian(f, x)` → optional; return a `Hessian` (here a dense `MatrixHessian`)

Then you register a **generator** under a symbol with `register_random_problem!`. After
that, `make_problem(RandomProblem(name = :ridge, params = …), rng)` produces instances.
`Problem` bundles the objective with the start point `x0`, an optional `x_opt` (which turns
on $\lVert x_k - x^\star\rVert$ tracking and the `DistanceToOptimal` criterion), and a free-form
`meta` dictionary.

In [ ]:
import .TestEngine: Objective, Hessian, MatrixHessian, Problem,
    value, grad!, hessian, register_random_problem!

# --- the objective: f(x) = ½‖Ax−b‖² + (μ/2)‖x‖² ---
struct RidgeObjective <: Objective
    A::Matrix{Float64}
    b::Vector{Float64}
    μ::Float64
end

value(f::RidgeObjective, x::Vector{Float64}) =
    0.5 * sum(abs2, f.A * x .- f.b) + 0.5 * f.μ * sum(abs2, x)

function grad!(g::Vector{Float64}, f::RidgeObjective, x::Vector{Float64})
    g .= f.A' * (f.A * x .- f.b) .+ f.μ .* x      # ∇f(x) = Aᵀ(Ax−b) + μx
    return g
end

hessian(f::RidgeObjective, x::Vector{Float64})::Hessian = MatrixHessian(f.A' * f.A + f.μ * I)

# --- register a generator under :ridge ---
register_random_problem!(:ridge, (rng, p) -> begin
    m  = get(p, :m, 120); n = get(p, :n, 40)
    μ  = get(p, :mu, 1.0e-2); κd = get(p, :condition_number, 100.0)
    # Controlled spectrum: singular values 1 → 1/√κd, so λ(AᵀA) ∈ [1/κd, 1] and
    # λ_max(AᵀA) = 1  ⇒  L = 1 + μ is known a priori (no per-instance L needed).
    U = Matrix(qr(randn(rng, m, n)).Q)[:, 1:n]
    V = Matrix(qr(randn(rng, n, n)).Q)
    s = exp10.(range(0, -0.5 * log10(κd); length = n))
    A = U * Diagonal(s) * V'
    b = randn(rng, m)
    L, msc = 1.0 + μ, minimum(s)^2 + μ
    x_opt  = (A' * A + μ * I) \ (A' * b)          # closed-form minimizer
    Problem(RidgeObjective(A, b, μ), zeros(n);
        meta  = Dict{Symbol,Any}(:L => L, :mu => μ, :m_sc => msc, :condition_number => L / msc),
        x_opt = x_opt)
end)

In [ ]:
# Sanity check: the registered family is now instantiable, and the gradient
# vanishes at the closed-form optimum.
prob = make_problem(RandomProblem(name = :ridge,
                                  params = (m = 120, n = 40, mu = 1.0e-2, condition_number = 100.0)),
                    Xoshiro(0))
@assert norm(grad(prob.f, prob.x_opt)) < 1e-8
(L = prob.meta[:L], κ = round(prob.meta[:condition_number], digits = 1),
 grad_at_opt = norm(grad(prob.f, prob.x_opt)))

## 3 · Author a custom method

A method is content too: subtype `IterativeMethod` and implement the runner contract —

- **`init_state(method, problem, rng)`** → a mutable state that composes the engine's
  `IterateGroup` (`x`, `gradient`, `x_prev`), `MetricsGroup` (`objective`, `gradient_norm`,
  `step_norm`, `dist_to_opt`), and `TimingGroup` (`core_time_ns`).
- **`step!(method, state, problem, iter, logger, rng)`** → advance one iteration *in place*.
  Wrap the numeric kernel in `@core_timed state begin … end` so the engine attributes
  **core** compute time separately from logging/plotting overhead.
- **`extract_log_entry(method, state, iter)`** → optional; the engine's default logs the
  canonical metrics. Override it to attach method-specific `extras`.

The runner fills `dist_to_opt` from `problem.x_opt` automatically, so you don't compute it.

We implement Polyak's **heavy ball**:

$$x_{k+1} = x_k - \alpha\, \nabla f(x_k) + \beta\,(x_k - x_{k-1}).$$

With $\beta=0$ this is plain gradient descent; $\beta>0$ adds momentum that accelerates
convergence on ill-conditioned strongly convex problems.

In [ ]:
import .TestEngine: init_state, step!, extract_log_entry

@kwdef struct HeavyBall <: IterativeMethod
    α::Float64
    β::Float64 = 0.0          # β = 0 ⇒ plain gradient descent
end

@kwdef mutable struct HeavyBallState
    iterate::IterateGroup     # x, gradient, x_prev   (shared engine group)
    metrics::MetricsGroup     # objective, gradient_norm, step_norm, dist_to_opt
    timing::TimingGroup       # core_time_ns
end

function init_state(m::HeavyBall, problem, rng::AbstractRNG)
    x0 = copy(problem.x0)
    g0 = similar(x0); grad!(g0, problem.f, x0)
    HeavyBallState(
        iterate = IterateGroup(x = x0, gradient = g0, x_prev = Float64[]),  # x_prev empty ⇒ "no previous"
        metrics = MetricsGroup(objective = total_objective(problem, x0),
                               gradient_norm = norm(g0), step_norm = 0.0, dist_to_opt = Inf),
        timing  = TimingGroup(core_time_ns = 0),
    )
end

function step!(m::HeavyBall, s::HeavyBallState, problem::Problem, iter::Int,
               logger::Logger, rng::AbstractRNG)
    g  = s.iterate.gradient                       # ∇f(x_k), from init / previous step
    xk = copy(s.iterate.x)                         # x_k (kept simple; gradient_descent.jl shows the no-alloc form)
    has_prev = !isempty(s.iterate.x_prev)
    @core_timed s begin                            # ← the engine times this kernel
        if has_prev
            @. s.iterate.x = xk - m.α * g + m.β * (xk - s.iterate.x_prev)
        else
            @. s.iterate.x = xk - m.α * g          # first step: no momentum term
        end
    end
    # roll x_prev ← x_k for the next iteration
    isempty(s.iterate.x_prev) ? (s.iterate.x_prev = xk) : copyto!(s.iterate.x_prev, xk)
    @core_timed s begin                            # refresh objective + gradient at x_{k+1}
        s.metrics.objective = total_objective(problem, s.iterate.x)
        grad!(s.iterate.gradient, problem.f, s.iterate.x)
    end
    s.metrics.gradient_norm = norm(s.iterate.gradient)
    s.metrics.step_norm     = norm(s.iterate.x .- xk)
    # (dist_to_opt is filled by the runner from problem.x_opt)
end

# Optional: override to record method-specific extras in each log row.
function extract_log_entry(m::HeavyBall, s::HeavyBallState, iter::Int)
    IterationLog(iter = iter, core_time_ns = s.timing.core_time_ns,
        objective = s.metrics.objective, gradient_norm = s.metrics.gradient_norm,
        step_norm = s.metrics.step_norm, dist_to_opt = s.metrics.dist_to_opt,
        extras = Dict{Symbol,Any}(:step_size => m.α, :momentum => m.β))
end

## 4 · Configure the experiment

An `ExperimentConfig` declares *what to compare*. Crucially, a method's **role** (baseline
vs experimental) is set here, by which list it goes in — it is **not** a property of the
method's type. We put plain `GradientDescent` in `baseline_methods`, and sweep our
`HeavyBall` over a momentum axis with a `VariantGrid` whose `role = :experimental`.

Because the ridge generator fixes $\lambda_{\max}(A^\top A)=1$, the textbook step
$\gamma = 1/L = 1/(1+\mu)$ is known without inspecting the instance.

In [ ]:
μ = 1.0e-2
γ = 1 / (1 + μ)                       # = 1/L, since λ_max(AᵀA) = 1 by construction

register_abbreviation!("HeavyBall", "HB")          # friendly legend/short names
register_abbreviation!("GradientDescent", "GD")

# A 1-axis variant grid: sweep the momentum β. β = 0.0 must reproduce the GD baseline.
hb_grid = VariantGrid(
    base_name = "HeavyBall",
    axes      = [ VariantAxis(:β, 0.0 => "0.0", 0.5 => "0.5", 0.9 => "0.9") ],
    builder   = (; β) -> HeavyBall(α = γ, β = β),
    role      = :experimental,
)

config = ExperimentConfig(
    name              = "ridge_walkthrough",
    problem_spec      = RandomProblem(name = :ridge,
                                      params = (m = 120, n = 40, mu = μ, condition_number = 100.0)),
    baseline_methods  = IterativeMethod[ GradientDescent(step_size = FixedStep(α = γ)) ],
    variant_grids     = VariantGrid[ hb_grid ],
    stopping_criteria = stop_when_any(MaxIterations(n = 3000), DistanceToOptimal(tol = 1e-8)),
    n_runs            = 1,
    seed              = 42,
)

In [ ]:
# `resolve_methods` shows how the config + grid route into the two comparison buckets.
baseline, experimental = resolve_methods(config)
(baseline = [n for (n, _) in baseline], experimental = [n for (n, _) in experimental])

## 5 · Run it

`run_experiment` generates the problem from the spec (deterministically, from `seed`), runs
every method under the stopping criteria, and returns an `ExperimentResult`.

In [ ]:
result  = run_experiment(config)
results = result.run_results[1].method_results
println("saved to: ", result.experiment_path, "\n")
for (name, mr) in sort(collect(results), by = first)
    println("  ", rpad(name, 20), "iters = ", lpad(mr.n_iters, 4), "   stop = ", mr.stop_reason)
end

## 6 · Persist & reload

`run_experiment` already wrote a full record — a `manifest.json`, per-method CSV logs, and a
`result.jld2` — under `logs/<date>/<NNN>/`. `load_experiment` reads it back, so a comparison
you ran today round-trips exactly later.

In [ ]:
loaded = load_experiment(result.experiment_path)
@assert Set(keys(loaded.run_results[1].method_results)) == Set(keys(results))
println(length(list_experiments()), " experiment(s) on disk; reloaded ",
        length(keys(loaded.run_results[1].method_results)), " methods from ", result.experiment_path)

## 7 · Analyze & plot

`to_dataframe` flattens the per-iteration logs (every method × iteration) into a tidy
`DataFrame`. We plot $\lVert x_k - x^\star\rVert$ (logged for free, since we gave the problem an
`x_opt`) on a log axis using the engine's own `PlotSpec` / `render_plot!`.

In [ ]:
df = to_dataframe(result)
first(df, 6)

In [ ]:
df.dist_plot = max.(df.dist_to_opt, 1e-12)          # floor for the log axis
spec = TestEngine.PlotSpec(                          # qualified: Makie also exports `PlotSpec`
    data = df, x = :iter, y = :dist_plot, group_by = :method_name,
    xlabel = "iteration  k", ylabel = "‖xₖ − x*‖", yscale = :log10,
    title  = "Ridge (κ ≈ 50): GD baseline vs HeavyBall(β) variants",
)
fig = Figure(size = (900, 560))
render_plot!(fig[1, 1], spec)
fig

**Read it.** `HeavyBall[β=0.0]` lands exactly on the `GradientDescent` baseline — the
sanity check that $\beta=0$ recovers plain gradient descent. Turning on momentum then
accelerates convergence on this ill-conditioned problem: in our run GD needed ~990
iterations to reach $\lVert x_k-x^\star\rVert\le 10^{-8}$, $\beta=0.5$ about 470, and
$\beta=0.9$ about 400 — momentum buying a ~2.5× reduction with one extra term in `step!`.

## 8 · Recap & where to go next

In one notebook you defined a **problem** and a **method** as content, swept the method in a
**variant grid**, declared **roles**, **ran**, **persisted/reloaded**, and **plotted** — the
whole loop, using only exported API.

To go further:

- **More problem kinds** — besides `register_random_problem!`, there's `register_analytic_problem!`
  (parameterized, deterministic) and `register_file_loader!` (load from disk), all driven through
  `make_problem`.
- **Richer methods** — compose the shipped components (`FixedStep`, `ArmijoLS`, `BarzilaiBorwein`,
  preconditioners, extrapolation). The full method-authoring reference is
  [`docs/src/extending.md`](docs/src/extending.md).
- **Honest measurement** — set `count_oracles = true` on the config to tally `value`/`grad`/`hvp`
  calls; the `core_time_ns` you wrapped with `@core_timed` is already separated from overhead.
- **Statistics** — raise `n_runs` and use `aggregate_runs` for medians/IQR across seeds.
- **Reliability** — `DebugConfig` checks (monotonicity, gradient bounds, numerical-gradient).
- The doc map ([`docs/ARCHITECTURE.md`](docs/ARCHITECTURE.md)) and the one-command figure
  reproduction (`julia --project reproduce.jl`).